# N10 — Text Baseline and LSTM

**Sessions:** 51–53  
**Objective:** Compare a sparse TF-IDF baseline with a sequence model and analyse disagreement cases.

## Task contract
Run top-to-bottom from a fresh kernel. Do not install packages inside the notebook. Record one error or misconception and complete the independent transfer before submission.


In [ ]:
from __future__ import annotations
import hashlib, json, platform, random
from pathlib import Path
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Python:", platform.python_version())
print("Working directory:", Path.cwd())


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

rng=np.random.default_rng(SEED); pos=['clear','helpful','works','fast']; neg=['confusing','broken','fails','slow']; neutral=['lesson','model','code','data']
texts=[]; labels=[]
for i in range(500):
    label=int(rng.random()>.5); words=list(rng.choice(neutral,3))+list(rng.choice(pos if label else neg,2))
    if rng.random()<.15: words=['not']+words; label=1-label
    rng.shuffle(words); texts.append(' '.join(words)); labels.append(label)
tr,va=train_test_split(np.arange(len(texts)),test_size=.25,stratify=labels,random_state=SEED)
base=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2))),('model',LogisticRegression(max_iter=1000))]).fit([texts[i] for i in tr],np.array(labels)[tr])
bpred=base.predict([texts[i] for i in va]); print('TF-IDF F1:',f1_score(np.array(labels)[va],bpred))


In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
torch.manual_seed(SEED); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokens=sorted(set(' '.join(texts).split())); vocab={'<PAD>':0,'<UNK>':1,**{t:i+2 for i,t in enumerate(tokens)}}
max_len=10
def encode(text):
    ids=[vocab.get(t,1) for t in text.split()][:max_len]; return ids+[0]*(max_len-len(ids))
X=torch.tensor([encode(t) for t in texts]); y=torch.tensor(labels)
loader=DataLoader(TensorDataset(X[tr],y[tr]),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(SEED))
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__(); self.emb=nn.Embedding(len(vocab),16,padding_idx=0); self.lstm=nn.LSTM(16,20,batch_first=True); self.fc=nn.Linear(20,2)
    def forward(self,x):
        _,(h,_)=self.lstm(self.emb(x)); return self.fc(h[-1])
model=LSTMClassifier().to(device); opt=torch.optim.Adam(model.parameters(),lr=.02); loss_fn=nn.CrossEntropyLoss()
for epoch in range(4):
    model.train()
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
model.eval();
with torch.no_grad(): lp=model(X[va].to(device)).argmax(1).cpu().numpy()
print('LSTM F1:',f1_score(np.array(labels)[va],lp))


In [ ]:
disagree=np.array(va)[bpred!=lp]
for idx in disagree[:8]: print({'text':texts[idx],'true':labels[idx],'tfidf':int(bpred[list(va).index(idx)]),'lstm':int(lp[list(va).index(idx)])})


## Independent transfer
Create an error taxonomy for negation, rare words, length, duplicates, and source. Change only one tokenisation/model factor and report which slices improve or regress.


## Fresh-run checklist

- [ ] Restart kernel and run all cells in order.
- [ ] Confirm assertions pass.
- [ ] Record package versions and seed.
- [ ] Save the required artifact with a relative path.
- [ ] Add an error-log entry and AI-use note.
- [ ] Explain one teacher-selected cell without reading it.
